## Installation des packages

In [ ]:
import subprocess
import sys

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "plotly",
]

def install (pack):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pack, "--quiet"])

for pack in packages:
    install(pack)

print("Package installes !")

## Verifications

In [ ]:
import os

fichiers_necessaires = [
    "datasets-project/datasets/orders.csv",
    "datasets-project/datasets/products.csv",
    "datasets-project/datasets/aisles.csv",
    "datasets-project/datasets/departments.csv",
    "datasets-project/datasets/order_products.csv",
]

for fichier in fichiers_necessaires:
    if os.path.exists(fichier):
        print(f"{fichier} trouve")
    else:
        print(f"{fichier} MANQUANT")

## Import des packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Imports OK !")

## Preprocessing des donnees

**Chargement de donnees**

In [ ]:

DATA_PATH = "datasets-project/datasets"

orders = pd.read_csv(os.path.join(DATA_PATH, "orders.csv"))
products = pd.read_csv(os.path.join(DATA_PATH, "products.csv"))
aisles = pd.read_csv(os.path.join(DATA_PATH, "aisles.csv"))
departments = pd.read_csv(os.path.join(DATA_PATH, "departments.csv"))
order_products = pd.read_csv(os.path.join(DATA_PATH, "order_products.csv"))

print("orders :", orders.shape)
print("products :", products.shape)
print("aisles :", aisles.shape)
print("departments :", departments.shape)
print("order_products :", order_products.shape)

**Exploration des fichiers**

In [ ]:
print("ORDERS")
display(orders.head(3))
print(orders.dtypes)

print("\nPRODUCTS")
display(products.head(3))

print("\nAISLES")
display(aisles.head(3))

print("\nDEPARTEMENTS")
display(departments.head(3))

print("\nORDER_PRODUCTS")
display(order_products.head(3))

**Correction des colonnes de orders**

In [ ]:
orders = pd.read_csv(os.path.join(DATA_PATH, "orders.csv"))

orders = orders.rename(columns={
    "order_number"           : "order_dow",
    "order_dow"              : "order_hour_of_day", 
    "order_hour_of_day"      : "days_since_prior_order",
    "days_since_prior_order" : "order_number"
})

orders = orders.drop(columns=["order_number"])

print(orders.columns.tolist())
print(orders.head(3))
print("\nNaN par colonne :")
print(orders.isnull().sum())

**Fusionnage**

In [ ]:
df = order_products.merge(orders, on="order_id", how="left")
df = df.merge(products, on="product_id", how="left")
df = df.merge(aisles, on="aisle_id", how="left")
df = df.merge(departments, on="department_id", how="left")

print("Shape après fusion :", df.shape)

## Nettoyage

In [ ]:
df["days_since_prior_order"] = df["days_since_prior_order"].fillna(0)

df = df.dropna(subset=[
    "user_id",
    "order_dow",
    "order_hour_of_day",
    "product_name",
    "aisle",
    "department",
    "reordered",
    "add_to_cart_order",
    "aisle_id", 
    "department_id"
])

df["user_id"] = df["user_id"].astype(int)
df["order_dow"] = df["order_dow"].astype(int)
df["order_hour_of_day"] = df["order_hour_of_day"].astype(int)
df["days_since_prior_order"] = df["days_since_prior_order"].astype(int)
df["product_id"] = df["product_id"].astype(int)
df["reordered"] = df["reordered"].astype(int)
df["add_to_cart_order"] = df["add_to_cart_order"].astype(int)
df["aisle_id"] = df["aisle_id"].astype(int)
df["department_id"] = df["department_id"].astype(int)

df = df.drop(columns=["eval_set"])

df = df.drop_duplicates()

**Verifications**

In [ ]:
print("Shape final :", df.shape)
print("NaN restants :", df.isnull().sum().sum())
print("Doublons restants :", df.duplicated().sum())
print("\nTypes finaux :")
print(df.dtypes)
display(df.head(5))

**Préparation des données**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

features = ['order_dow', 'order_hour_of_day', 'days_since_prior_order', 'add_to_cart_order']
X = df[features]
y = df['reordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Données prêtes pour l'entraînement !")

**Régression Logistique**

In [ ]:
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

print("--- Évaluation Régression Logistique ---")
print(classification_report(y_test, y_pred_log))

**Random Forest**

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("--- Évaluation Random Forest ---")
print(classification_report(y_test, y_pred_rf))

**Comparaison**

In [ ]:
importances = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='importance', y='feature', data=importances, palette='viridis')
plt.title("Quels facteurs prédisent le mieux le réachat ?")
plt.show()

In [ ]:
df["is_organic"] = df["product_name"].str.contains("Organic", case=False, na=False).astype(int)

In [ ]:
from sklearn.cluster import KMeans

user_profiles = df.groupby('user_id').agg({
    'order_id': 'nunique',                
    'add_to_cart_order': 'mean',          
    'days_since_prior_order': 'mean',     
    'is_organic': 'mean'                  
}).rename(columns={'order_id': 'frequency', 'avg_basket_size': 'avg_basket'})


from sklearn.preprocessing import StandardScaler
scaler_k = StandardScaler()
user_profiles_scaled = scaler_k.fit_transform(user_profiles)


kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
user_profiles['cluster'] = kmeans.fit_predict(user_profiles_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=user_profiles, x='frequency', y='is_organic', hue='cluster', palette='viridis')
plt.title("Segmentation Clients : Fréquence vs Appétence Bio")
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100))
])

pipeline.fit(X_train, y_train)